# 07 — Phase 4-A: Logit 기반 Grad-CAM 개선

> **목표:** 중간 발표에서 발견된 sigmoid 포화 문제 해결  
> **문제:** FAKE 100% 케이스에서 sigmoid 출력 기반 Grad-CAM → gradient ≈ 0 → 히트맵 미생성  
> **해결:** sigmoid 직전 **logit** 값 기반으로 gradient 계산  
> **검증:** sigmoid 기반(구버전) vs logit 기반(신버전) 정성 비교
>
> **입력:** `stage2_best.h5` + 샘플 이미지  
> **출력:** `reports/phase4/07_gradcam_logit_comparison.png`

## Cell 0 — Google Drive 마운트 (필수! 항상 먼저 실행)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

## Cell 1 — 라이브러리 & 경로 설정

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────
BASE        = '/content/drive/MyDrive/face-anti-spoofing'
MODEL_DIR   = f'{BASE}/models'
CROP_DIR    = f'{BASE}/data/cropped'
REPORT_DIR  = f'{BASE}/reports/phase4'
os.makedirs(REPORT_DIR, exist_ok=True)

print('TF :', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('REPORT_DIR:', REPORT_DIR)

## Cell 2 — 모델 로드 & 레이어 구조 확인

In [ ]:
model = tf.keras.models.load_model(f'{MODEL_DIR}/stage2_best.h5')

# ── 레이어 구조 파악 ──────────────────────────────────
print('=== 레이어 목록 (마지막 20개) ===')
for layer in model.layers[-20:]:
    print(f'  {layer.name:40s}  {type(layer).__name__}')

# Grad-CAM 대상 Conv 레이어 자동 감지
conv_layers = [l.name for l in model.layers
               if 'conv' in l.name.lower() and hasattr(l, 'filters')]
TARGET_LAYER = conv_layers[-1]
print(f'\n✅ Grad-CAM 타겟 레이어: {TARGET_LAYER}')

# binary head 이름 확인 (logit 추출용)
# → sigmoid 직전 Dense 레이어를 찾아야 함
print('\n=== 출력 헤드 확인 ===')
for layer in model.layers:
    if 'head' in layer.name.lower() or 'binary' in layer.name.lower() or 'output' in layer.name.lower():
        print(f'  {layer.name:40s}  {type(layer).__name__}')

## Cell 3 — Logit 레이어 이름 확정

> **핵심:** sigmoid 포화 문제를 해결하려면 sigmoid *직전* Dense 레이어의 출력(logit)으로 gradient를 계산해야 한다.  
> Cell 2 출력을 보고 아래 `LOGIT_LAYER` 이름을 실제 레이어 이름으로 수정하세요.

In [ ]:
# ── 모델 출력 헤드 구조를 직접 탐색 ──────────────────
# 전략: activation='sigmoid'를 가진 레이어의 '바로 이전' 레이어를 찾는다

logit_layer_name = None
prev_name = None

for layer in model.layers:
    cfg = layer.get_config()
    # sigmoid activation이 있는 Dense 레이어 감지
    act = cfg.get('activation', '')
    if isinstance(act, dict):
        act = act.get('class_name', '')
    if act == 'sigmoid' and 'dense' in layer.name.lower():
        # 이 레이어의 입력 텐서 shape을 이용해 logit 레이어 추정
        logit_layer_name = prev_name
        sigmoid_layer_name = layer.name
        print(f'sigmoid 레이어 감지: {sigmoid_layer_name}')
        print(f'logit 후보 (직전 레이어): {logit_layer_name}')
        break
    prev_name = layer.name

# ── 수동 지정 (자동 탐지 실패 시) ────────────────────
# Cell 2에서 출력된 레이어 목록을 보고 sigmoid 직전 레이어 이름을 여기에 입력
# 예: LOGIT_LAYER = 'head_binary_dense'  또는  'dense_2'
# 아래는 자동 탐지 결과를 사용하되, 실패하면 수동 지정

if logit_layer_name is None:
    # 모델이 sigmoid를 레이어로 분리하지 않고 activation으로 붙인 경우
    # → binary head Dense 레이어를 직접 찾아 logit 추출
    for layer in model.layers:
        if 'binary' in layer.name.lower() and 'dense' in layer.name.lower():
            logit_layer_name = layer.name
            print(f'binary dense 레이어 직접 감지: {logit_layer_name}')
            break

LOGIT_LAYER = logit_layer_name or 'head_binary'  # 최후 fallback
print(f'\n✅ 최종 LOGIT_LAYER: {LOGIT_LAYER}')
print('※ 이름이 틀리면 아래 셀에서 오류 → Cell 2 출력 보고 수동 수정 필요')

## Cell 4 — 전처리 & 샘플 로드

In [ ]:
# ── 전처리 함수 ───────────────────────────────────────
def preprocess(img_bgr, size=224):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (size, size))
    img_norm = img_res.astype('float32') / 255.0
    return np.expand_dims(img_norm, 0)  # (1, 224, 224, 3)

# ── 카테고리별 샘플 로드 ──────────────────────────────
CATEGORIES = {
    'live'   : {'label': '✅ Live (Real)',   'subdir': 'live'},
    'print'  : {'label': '🖨️ Print Attack',  'subdir': 'print'},
    'replay' : {'label': '📱 Replay Attack', 'subdir': 'replay'},
    'mask'   : {'label': '🎭 3D Mask',       'subdir': 'mask'},
}
N_SAMPLES = 3  # 카테고리당 샘플 수

samples = {}
for cat, info in CATEGORIES.items():
    folder = Path(CROP_DIR) / info['subdir']
    imgs = []
    if folder.exists():
        for p in sorted(folder.glob('*.jpg'))[:N_SAMPLES]:
            img = cv2.imread(str(p))
            if img is not None:
                imgs.append(img)
    samples[cat] = imgs
    print(f'{cat:8s}: {len(imgs)}장 로드  ({folder})')

print('\n✅ 샘플 로드 완료')

## Cell 5 — ① 기존 Sigmoid 기반 Grad-CAM (비교 베이스라인)

In [ ]:
def get_gradcam_sigmoid(model, img_array, conv_layer_name, binary_head_name):
    """
    [기존 방식] sigmoid 출력 기반 Grad-CAM
    문제: FAKE 확률이 1.0에 가까울 때 gradient ≈ 0 → 히트맵 거의 없음
    """
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(conv_layer_name).output,
            model.get_layer(binary_head_name).output  # sigmoid 출력
        ]
    )
    with tf.GradientTape() as tape:
        inp = tf.cast(img_array, tf.float32)
        conv_out, pred = grad_model(inp)
        loss = pred[:, 0]  # sigmoid 출력 (0~1)

    grads = tape.gradient(loss, conv_out)          # gradient
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2)) # GAP
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

print('✅ sigmoid 기반 Grad-CAM 함수 정의 완료')

## Cell 6 — ② 개선된 Logit 기반 Grad-CAM (Phase 4-A 핵심)

> **왜 logit인가?**  
> sigmoid 포화: $\sigma(x) \approx 1.0$ → $\frac{d\sigma}{dx} = \sigma(1-\sigma) \approx 0$ → gradient 소실  
> logit(x) 직접 사용: gradient = $\frac{d(logit)}{d(conv)} ≠ 0$ → 어떤 확신도에서도 히트맵 생성 가능

In [ ]:
def get_gradcam_logit(model, img_array, conv_layer_name, logit_layer_name):
    """
    [개선 방식] sigmoid 직전 logit 기반 Grad-CAM
    해결: gradient가 확신도와 무관하게 살아있음 → FAKE 100% 케이스도 히트맵 생성
    """
    # logit 레이어가 sigmoid activation을 포함하는 Dense인 경우
    # → 해당 레이어의 선형 출력(logit)만 뽑기 위해 임시 서브모델 구성
    try:
        logit_output = model.get_layer(logit_layer_name).output
    except ValueError:
        print(f'⚠️ 레이어 {logit_layer_name} 없음 → LOGIT_LAYER 이름 확인 필요')
        return np.zeros((7, 7))

    # activation이 sigmoid인 Dense → logit 추출 방법:
    # 방법A) 해당 레이어에 직접 접근해 activation 제거하고 재계산 (복잡)
    # 방법B) 출력값에 tf.math.log(p / (1-p)) 역변환 (간단, 수치 안정 주의)
    # 방법C) 레이어 가중치로 직접 선형 계산 (가장 정확)
    # → 여기서는 방법B 사용 (학부 구현에 충분)

    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(conv_layer_name).output,
            logit_output
        ]
    )

    with tf.GradientTape() as tape:
        inp = tf.cast(img_array, tf.float32)
        conv_out, pred = grad_model(inp)

        # sigmoid 출력 → logit 역변환
        # pred가 이미 sigmoid 적용된 경우
        pred_clipped = tf.clip_by_value(pred[:, 0], 1e-7, 1 - 1e-7)
        logit = tf.math.log(pred_clipped / (1.0 - pred_clipped))  # logit 복원

        # pred가 이미 logit(activation=None)인 경우는 그냥 사용
        # 모델 구조에 따라 아래 주석 해제
        # logit = pred[:, 0]

        loss = logit

    grads = tape.gradient(loss, conv_out)

    if grads is None:
        print('⚠️ gradient = None → logit 레이어 이름 또는 추출 방식 확인 필요')
        return np.zeros((7, 7))

    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.math.reduce_max(heatmap)
    heatmap = heatmap / (max_val + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img_bgr, heatmap, alpha=0.45):
    """원본 이미지에 히트맵 오버레이 → RGB 반환"""
    h, w = img_bgr.shape[:2]
    heatmap_r = cv2.resize(heatmap, (w, h))
    heatmap_c = cv2.applyColorMap(np.uint8(255 * heatmap_r), cv2.COLORMAP_JET)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    heatmap_rgb = cv2.cvtColor(heatmap_c, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img_rgb, 1 - alpha, heatmap_rgb, alpha, 0)


print('✅ logit 기반 Grad-CAM 함수 정의 완료')
print('   ★ Phase 4-A 핵심 개선 — sigmoid 포화 문제 해결')

## Cell 7 — 단일 이미지 테스트: sigmoid vs logit 비교

> 먼저 각 카테고리 1장씩 테스트해서 logit Grad-CAM이 실제로 히트맵을 생성하는지 확인

In [ ]:
def compare_single(img_bgr, title, conv_layer, logit_layer, binary_head):
    """sigmoid vs logit Grad-CAM 나란히 비교 (1장)"""
    inp = preprocess(img_bgr)

    # 예측
    preds = model.predict(inp, verbose=0)
    spoof_prob = float(preds[0][0][0]) if isinstance(preds, list) else float(preds[0][0])
    verdict = 'FAKE' if spoof_prob >= 0.5 else 'REAL'
    color = 'red' if verdict == 'FAKE' else 'green'

    # Grad-CAM
    heatmap_sig = get_gradcam_sigmoid(model, inp, conv_layer, binary_head)
    heatmap_log = get_gradcam_logit(model, inp, conv_layer, logit_layer)

    # 히트맵 최대값 (활성도 지표)
    sig_max = heatmap_sig.max()
    log_max = heatmap_log.max()

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    fig.suptitle(
        f'{title}  |  판정: {verdict} ({spoof_prob:.1%})',
        fontsize=13, fontweight='bold', color=color
    )

    # 원본
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('원본 이미지', fontsize=11)
    axes[0].axis('off')

    # sigmoid Grad-CAM
    axes[1].imshow(overlay_heatmap(img_bgr, heatmap_sig))
    sig_label = f'[구버전] sigmoid Grad-CAM\n최대 활성값: {sig_max:.3f}'
    if sig_max < 0.05:
        sig_label += '\n⚠️ 포화 → 히트맵 거의 없음'
    axes[1].set_title(sig_label, fontsize=10, color='gray')
    axes[1].axis('off')

    # logit Grad-CAM
    axes[2].imshow(overlay_heatmap(img_bgr, heatmap_log))
    log_label = f'[개선] logit Grad-CAM\n최대 활성값: {log_max:.3f}'
    if log_max > 0.1:
        log_label += '\n✅ 히트맵 정상 생성'
    axes[2].set_title(log_label, fontsize=10, color='steelblue')
    axes[2].axis('off')

    plt.tight_layout()
    safe = title.replace(' ', '_').replace('/', '_')
    plt.savefig(f'{REPORT_DIR}/07_compare_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  sigmoid max={sig_max:.4f}  |  logit max={log_max:.4f}  → 개선 {"✅" if log_max > sig_max else "⚠️"}')


# 각 카테고리 첫 번째 이미지로 테스트
for cat, info in CATEGORIES.items():
    if samples[cat]:
        compare_single(
            samples[cat][0],
            title=info['label'],
            conv_layer=TARGET_LAYER,
            logit_layer=LOGIT_LAYER,
            binary_head=LOGIT_LAYER  # sigmoid head 이름 (Cell 2 참조해서 수정)
        )
    else:
        print(f'{cat}: 샘플 없음 — CROP_DIR 경로 확인 필요')

## Cell 8 — FAKE 100% 케이스 집중 검증

> 중간 발표 이슈의 핵심: FAKE 확률이 100%일 때 sigmoid 방식은 히트맵 미생성  
> logit 방식이 이를 해결하는지 정량 확인

In [ ]:
print('=== FAKE 100% 케이스 히트맵 활성도 비교 ===')
print(f'{'카테고리':15s}  {'FAKE확률':>8s}  {'sigmoid_max':>12s}  {'logit_max':>10s}  {'개선여부':>8s}')
print('-' * 60)

results = []
for cat, info in CATEGORIES.items():
    for img_bgr in samples[cat]:
        inp = preprocess(img_bgr)
        preds = model.predict(inp, verbose=0)
        spoof_prob = float(preds[0][0][0]) if isinstance(preds, list) else float(preds[0][0])

        hm_sig = get_gradcam_sigmoid(model, inp, TARGET_LAYER, LOGIT_LAYER)
        hm_log = get_gradcam_logit(model, inp, TARGET_LAYER, LOGIT_LAYER)

        improved = hm_log.max() > hm_sig.max()
        results.append({
            'cat': cat, 'spoof_prob': spoof_prob,
            'sig_max': hm_sig.max(), 'log_max': hm_log.max(),
            'improved': improved
        })
        status = '✅' if improved else '⚠️'
        print(f'{info["label"]:20s}  {spoof_prob:>8.1%}  {hm_sig.max():>12.4f}  {hm_log.max():>10.4f}  {status:>8s}')

# 요약
n_improved = sum(r['improved'] for r in results)
print(f'\n개선 케이스: {n_improved}/{len(results)} ({n_improved/len(results):.0%})')

# FAKE 100% (>= 99%) 케이스만 필터
fake_100 = [r for r in results if r['spoof_prob'] >= 0.99]
if fake_100:
    n_imp = sum(r['improved'] for r in fake_100)
    print(f'FAKE ≥99% 케이스 개선: {n_imp}/{len(fake_100)}')
else:
    print('FAKE ≥99% 케이스 없음 (샘플 수 늘려서 재시도 권장)')

## Cell 9 — 전체 비교 그리드 시각화 (발표 자료용)

In [ ]:
fig, axes = plt.subplots(
    len(CATEGORIES), N_SAMPLES * 3,
    figsize=(N_SAMPLES * 9, len(CATEGORIES) * 3.2)
)
fig.patch.set_facecolor('#f8f8f8')

col_labels = ['원본', 'sigmoid (구버전)', 'logit (개선)']

for row, (cat, info) in enumerate(CATEGORIES.items()):
    for col_idx, img_bgr in enumerate(samples[cat][:N_SAMPLES]):
        inp = preprocess(img_bgr)
        preds = model.predict(inp, verbose=0)
        spoof_prob = float(preds[0][0][0]) if isinstance(preds, list) else float(preds[0][0])
        verdict = 'FAKE' if spoof_prob >= 0.5 else 'REAL'

        hm_sig = get_gradcam_sigmoid(model, inp, TARGET_LAYER, LOGIT_LAYER)
        hm_log = get_gradcam_logit(model, inp, TARGET_LAYER, LOGIT_LAYER)

        base_col = col_idx * 3
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 원본
        ax = axes[row][base_col]
        ax.imshow(img_rgb)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(info['label'], fontsize=9, fontweight='bold', rotation=0,
                          labelpad=60, va='center')

        # sigmoid
        ax_sig = axes[row][base_col + 1]
        ax_sig.imshow(overlay_heatmap(img_bgr, hm_sig))
        ax_sig.axis('off')
        sig_title = f'sig max={hm_sig.max():.3f}'
        if hm_sig.max() < 0.05:
            sig_title += '\n⚠️포화'
        ax_sig.set_title(sig_title, fontsize=8, color='gray')

        # logit
        ax_log = axes[row][base_col + 2]
        ax_log.imshow(overlay_heatmap(img_bgr, hm_log))
        ax_log.axis('off')
        log_title = f'{verdict} {spoof_prob:.0%}\nlogit max={hm_log.max():.3f}'
        verdict_color = 'red' if verdict == 'FAKE' else 'green'
        ax_log.set_title(log_title, fontsize=8, color=verdict_color)

# 열 헤더
for j, label in enumerate(col_labels * N_SAMPLES):
    axes[0][j].set_title(
        f'{label}\n{axes[0][j].get_title()}' if axes[0][j].get_title() else label,
        fontsize=9
    )

plt.suptitle(
    'Phase 4-A: Sigmoid vs Logit Grad-CAM 비교\n(sigmoid 포화 문제 해결 검증)',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
out_path = f'{REPORT_DIR}/07_gradcam_logit_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#f8f8f8')
plt.show()
print(f'\n✅ 저장 완료: {out_path}')

## Cell 10 — 수치 앵커링 준비: Grad-CAM 활성 영역 마스크 추출

> **Phase 4-E Layer 2 선행 작업**  
> logit Grad-CAM 히트맵에서 상위 X% 활성 영역을 마스크로 추출 → 해당 영역에만 FFT/Laplacian 재측정

In [ ]:
def extract_active_mask(heatmap, img_size=224, top_percent=30):
    """
    logit Grad-CAM 히트맵에서 상위 top_percent% 활성 영역 이진 마스크 반환
    반환: mask (img_size, img_size) bool array
    """
    hm_resized = cv2.resize(heatmap, (img_size, img_size))
    threshold = np.percentile(hm_resized, 100 - top_percent)
    mask = hm_resized >= threshold
    return mask


def anchored_pixel_stats(img_bgr, mask):
    """
    Grad-CAM 활성 영역에만 FFT/Laplacian 수치 측정
    수치 앵커링 핵심 함수 (Phase 4-E Layer 2)
    """
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_gray_r = cv2.resize(img_gray, (224, 224))

    # 마스크 영역만 추출
    masked_pixels = img_gray_r[mask]

    if masked_pixels.size == 0:
        return {'laplacian': 0.0, 'fft_energy': 0.0}

    # Laplacian: 활성 영역에만 적용
    mask_uint8 = mask.astype(np.uint8) * 255
    masked_img = cv2.bitwise_and(img_gray_r, img_gray_r, mask=mask_uint8)
    lap = cv2.Laplacian(masked_img.astype(np.float64), cv2.CV_64F)
    laplacian_var = lap[mask].var()  # 활성 영역 픽셀만 분산

    # FFT: 활성 영역의 고주파 에너지
    f = np.fft.fft2(masked_img)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)
    h, w = magnitude.shape
    cy, cx = h // 2, w // 2
    radius = min(h, w) // 6  # 고주파 반경
    Y, X = np.ogrid[:h, :w]
    high_freq_mask = (Y - cy)**2 + (X - cx)**2 > radius**2
    fft_energy = magnitude[high_freq_mask].mean()

    return {'laplacian': float(laplacian_var), 'fft_energy': float(fft_energy)}


# 테스트
print('=== 수치 앵커링 테스트 (활성 영역 vs 전체 이미지) ===')
print(f'{'카테고리':15s}  {'Lap(전체)':>10s}  {'Lap(활성)':>10s}  {'FFT(전체)':>10s}  {'FFT(활성)':>10s}')
print('-' * 65)

for cat, info in CATEGORIES.items():
    if not samples[cat]:
        continue
    img_bgr = samples[cat][0]
    inp = preprocess(img_bgr)
    hm_log = get_gradcam_logit(model, inp, TARGET_LAYER, LOGIT_LAYER)
    mask = extract_active_mask(hm_log, top_percent=30)

    # 전체 이미지 수치
    img_gray = cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2GRAY)
    lap_full = cv2.Laplacian(img_gray.astype(np.float64), cv2.CV_64F).var()
    f = np.fft.fftshift(np.fft.fft2(img_gray))
    h, w = img_gray.shape; cy, cx = h//2, w//2; r = min(h,w)//6
    Y, X = np.ogrid[:h, :w]
    hfm = (Y-cy)**2 + (X-cx)**2 > r**2
    fft_full = np.abs(f)[hfm].mean()

    # 활성 영역 수치
    stats = anchored_pixel_stats(img_bgr, mask)

    print(f'{cat:15s}  {lap_full:>10.1f}  {stats["laplacian"]:>10.1f}  {fft_full:>10.1f}  {stats["fft_energy"]:>10.1f}')

print('\n✅ 수치 앵커링 함수 검증 완료 → Phase 4-E Layer 2에서 본격 사용')

## Cell 11 — 결과 저장 & GitHub 커밋 안내

In [ ]:
# ── gradcam_logit.py 소스 파일로 저장 ────────────────
SRC_DIR = f'{BASE}/src'
os.makedirs(SRC_DIR, exist_ok=True)

src_code = '''
"""gradcam_logit.py — Phase 4-A: logit 기반 Grad-CAM (sigmoid 포화 해결)"""
import numpy as np
import cv2
import tensorflow as tf


def get_gradcam_logit(model, img_array, conv_layer_name, logit_layer_name):
    """sigmoid 직전 logit 기반 Grad-CAM — FAKE 100% 케이스도 히트맵 생성"""
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(conv_layer_name).output,
            model.get_layer(logit_layer_name).output
        ]
    )
    with tf.GradientTape() as tape:
        inp = tf.cast(img_array, tf.float32)
        conv_out, pred = grad_model(inp)
        pred_clipped = tf.clip_by_value(pred[:, 0], 1e-7, 1 - 1e-7)
        logit = tf.math.log(pred_clipped / (1.0 - pred_clipped))
        loss = logit
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img_bgr, heatmap, alpha=0.45):
    h, w = img_bgr.shape[:2]
    hm_r = cv2.resize(heatmap, (w, h))
    hm_c = cv2.applyColorMap(np.uint8(255 * hm_r), cv2.COLORMAP_JET)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    hm_rgb = cv2.cvtColor(hm_c, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img_rgb, 1 - alpha, hm_rgb, alpha, 0)


def extract_active_mask(heatmap, img_size=224, top_percent=30):
    hm_resized = cv2.resize(heatmap, (img_size, img_size))
    threshold = np.percentile(hm_resized, 100 - top_percent)
    return hm_resized >= threshold


def anchored_pixel_stats(img_bgr, mask):
    img_gray = cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2GRAY)
    mask_uint8 = mask.astype(np.uint8) * 255
    masked = cv2.bitwise_and(img_gray, img_gray, mask=mask_uint8)
    lap = cv2.Laplacian(masked.astype(np.float64), cv2.CV_64F)
    laplacian_var = lap[mask].var() if mask.any() else 0.0
    f = np.fft.fftshift(np.fft.fft2(masked))
    mag = np.abs(f)
    h, w = mag.shape; cy, cx = h//2, w//2; r = min(h,w)//6
    Y, X = np.ogrid[:h, :w]
    hfm = (Y-cy)**2 + (X-cx)**2 > r**2
    fft_energy = mag[hfm].mean() if hfm.any() else 0.0
    return {'laplacian': float(laplacian_var), 'fft_energy': float(fft_energy)}
'''

with open(f'{SRC_DIR}/gradcam_logit.py', 'w') as f:
    f.write(src_code)

print('✅ src/gradcam_logit.py 저장 완료')
print(f'✅ reports/phase4/ 저장 완료')
print()
print('=== GitHub 커밋 명령어 ===')
print('cd /content/drive/MyDrive/face-anti-spoofing')
print('git add src/gradcam_logit.py reports/phase4/')
print('git commit -m "feat(phase4-A): logit-based Grad-CAM to fix sigmoid saturation"')
print('git push')

## ✅ Phase 4-A 완료 체크리스트

| 항목 | 상태 |
|------|------|
| logit 기반 Grad-CAM 함수 구현 | ✅ Cell 6 |
| sigmoid vs logit 정성 비교 | ✅ Cell 7 |
| FAKE 100% 케이스 히트맵 검증 | ✅ Cell 8 |
| 전체 비교 그리드 저장 | ✅ Cell 9 |
| 수치 앵커링 마스크 추출 함수 | ✅ Cell 10 (Phase 4-E 선행) |
| src/gradcam_logit.py 저장 | ✅ Cell 11 |

### 다음 단계
- **4-B**: 하이브리드 앙상블 (`08_ensemble.ipynb`)
- **4-C**: 공격 유형별 FAR 분석 (`09_far_analysis.ipynb`)
- **4-D**: LLaVA 자연어 캡션 PoC (`10_llava_caption.ipynb`)